In [ ]:
# terminal
# XXX is the spectrograms.zip id from google drive
pip install gdown
gdown --id 180AhoIMN0jXr8_6-ozoIESH6rpIQVUGt
apt-get update && apt-get install -y unzip
unzip spectrograms.zip
apt-get update && apt-get install -y ffmpeg

In [ ]:
# terminal
# may need to run this if cannot install unzip
apt-get update -o Dir::Etc::sourcelist="sources.list.d/ubuntu.sources" -o Dir::Etc::sourceparts="-" -o APT::Get::List-Cleanup="0"
apt-get install -y unzip

In [ ]:
!pip install librosa
!pip install pydub
!pip install pandas
!pip install torchmetrics
!pip install matplotlib

In [ ]:
import sys
!git clone https://github.com/majfu/MusicGenreClassifier.git
repo_path = "/workspace/MusicGenreClassifier"
sys.path.append(repo_path)
sys.path.insert(0, repo_path)

In [3]:
import os

# train and val labels df needs to be uploaded to the DATA_DIR
DATA_DIR = "/workspace/data"
MODELS_DIRECTORY_RP = "/workspace/models/"

MEAN_TRAIN_RP = os.path.join(DATA_DIR, "train_mean.pt")
STD_TRAIN_RP = os.path.join(DATA_DIR, "train_std.pt")
MEAN_VAL_RP = os.path.join(DATA_DIR, "val_mean.pt")
STD_VAL_RP = os.path.join(DATA_DIR, "val_std.pt")

TRAIN_SPLIT_RP = os.path.join(DATA_DIR, "train.csv")
VAL_SPLIT_RP = os.path.join(DATA_DIR, "val.csv")

SPECTROGRAMS_RP = "/workspace/spectrograms"
PLOTS_RP = "/workspace/plots"

In [6]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
import torch.nn.functional as F
from src.features.standardization import StandardizationTransform
from src.data.feature_dataset import FeatureDataset
from src.utils.io_utils import *
from src.utils.metadata_utils import *
from torchmetrics.classification import MultilabelF1Score, MultilabelPrecision, MultilabelRecall
import matplotlib.pyplot as plt

In [7]:
from src.data.feature_dataset import FeatureDataset
from src.utils.io_utils import calculate_and_save_dataset_mean_and_std

train_fds = FeatureDataset(TRAIN_SPLIT_RP, SPECTROGRAMS_RP)
val_fds = FeatureDataset(VAL_SPLIT_RP, SPECTROGRAMS_RP)

# after computing them once download them onto permanent disk
calculate_and_save_dataset_mean_and_std(train_fds, MEAN_TRAIN_RP, STD_TRAIN_RP)
calculate_and_save_dataset_mean_and_std(val_fds, MEAN_VAL_RP, STD_VAL_RP)

In [48]:
class MusicGenreCNN(nn.Module):
    def __init__(self, num_classes, hidden_size=512):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(3, 3), padding='same', stride=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 2))
        )
        self.conv2 = nn.Sequential(
            nn.Dropout(0.2),
            nn.Conv2d(32, 64, kernel_size=(3, 3), padding='same', stride=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 2))
        )
        self.conv3 = nn.Sequential(
            nn.Dropout(0.2),
            nn.Conv2d(64, 128, kernel_size=(3, 3), padding='same', stride=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d((2, 2))
        )
        self.conv4 = nn.Sequential(
            nn.Dropout(0.2),
            nn.Conv2d(128, 256, kernel_size=(3, 3), padding='same', stride=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d((2, 2))
        )
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.flatten = nn.Flatten()
        self.average_pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(256, hidden_size)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.average_pooling(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [36]:
# this class is taken from https://github.com/itakurah/Focal-loss-PyTorch/blob/main/focal_loss.py

class FocalLoss(nn.Module):
    def __init__(self, gamma=2, alpha=0.5, reduction='batch_mean', task_type='multi-label', num_classes=None):
        """
        Unified Focal Loss class for binary, multi-class, and multi-label classification tasks.
        :param gamma: Focusing parameter, controls the strength of the modulating factor (1 - p_t)^gamma
        :param alpha: Balancing factor, can be a scalar or a tensor for class-wise weights. If None, no class balancing is used.
        :param reduction: Specifies the reduction method: 'none' | 'mean' | 'sum'
        :param task_type: Specifies the type of task: 'binary', 'multi-class', or 'multi-label'
        :param num_classes: Number of classes (only required for multi-class classification)
        """
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction
        self.task_type = task_type
        self.num_classes = num_classes

        # Handle alpha for class balancing in multi-class tasks
        if task_type == 'multi-class' and alpha is not None and isinstance(alpha, (list, torch.Tensor)):
            assert num_classes is not None, "num_classes must be specified for multi-class classification"
            if isinstance(alpha, list):
                self.alpha = torch.Tensor(alpha)
            else:
                self.alpha = alpha

    def forward(self, inputs, targets):
        """
        Forward pass to compute the Focal Loss based on the specified task type.
        :param inputs: Predictions (logits) from the model.
                       Shape:
                         - binary/multi-label: (batch_size, num_classes)
                         - multi-class: (batch_size, num_classes)
        :param targets: Ground truth labels.
                        Shape:
                         - binary: (batch_size,)
                         - multi-label: (batch_size, num_classes)
                         - multi-class: (batch_size,)
        """
        if self.task_type == 'binary':
            return self.binary_focal_loss(inputs, targets)
        elif self.task_type == 'multi-class':
            return self.multi_class_focal_loss(inputs, targets)
        elif self.task_type == 'multi-label':
            return self.multi_label_focal_loss(inputs, targets)
        else:
            raise ValueError(
                f"Unsupported task_type '{self.task_type}'. Use 'binary', 'multi-class', or 'multi-label'.")

    def binary_focal_loss(self, inputs, targets):
        """ Focal loss for binary classification. """
        probs = torch.sigmoid(inputs)
        targets = targets.float()

        # Compute binary cross entropy
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')

        # Compute focal weight
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_weight = (1 - p_t) ** self.gamma

        # Apply alpha if provided
        if self.alpha is not None:
            alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
            bce_loss = alpha_t * bce_loss

        # Apply focal loss weighting
        loss = focal_weight * bce_loss

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

    def multi_class_focal_loss(self, inputs, targets):
        """ Focal loss for multi-class classification. """
        if self.alpha is not None:
            alpha = self.alpha.to(inputs.device)

        # Convert logits to probabilities with softmax
        probs = F.softmax(inputs, dim=1)

        # One-hot encode the targets
        targets_one_hot = F.one_hot(targets, num_classes=self.num_classes).float()

        # Compute cross-entropy for each class
        ce_loss = -targets_one_hot * torch.log(probs)

        # Compute focal weight
        p_t = torch.sum(probs * targets_one_hot, dim=1)  # p_t for each sample
        focal_weight = (1 - p_t) ** self.gamma

        # Apply alpha if provided (per-class weighting)
        if self.alpha is not None:
            alpha_t = alpha.gather(0, targets)
            ce_loss = alpha_t.unsqueeze(1) * ce_loss

        # Apply focal loss weight
        loss = focal_weight.unsqueeze(1) * ce_loss

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

    def multi_label_focal_loss(self, inputs, targets):
        """ Focal loss for multi-label classification. """
        probs = torch.sigmoid(inputs)

        # Compute binary cross entropy
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')

        # Compute focal weight
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_weight = (1 - p_t) ** self.gamma

        # Apply alpha if provided
        if self.alpha is not None:
            alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
            bce_loss = alpha_t * bce_loss

        # Apply focal loss weight
        loss = focal_weight * bce_loss
        if self.reduction == "batch_mean":
            return loss.sum(dim=1).mean()
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

In [66]:

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP_DTYPE = torch.bfloat16

def train(model, num_epochs, train_dl, val_dl, num_genres, genre_names, version_num, threshold, save_every=50):
    criterion = FocalLoss(num_classes=num_genres)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5
    )

    train_f1_macro = MultilabelF1Score(num_labels=num_genres, average='macro').to(DEVICE)
    val_f1_macro = MultilabelF1Score(num_labels=num_genres, average='macro').to(DEVICE)
    val_f1 = MultilabelF1Score(num_labels=num_genres, average=None).to(DEVICE)
    val_precision = MultilabelPrecision(num_labels=num_genres, average=None).to(DEVICE)
    val_recall = MultilabelRecall(num_labels=num_genres, average=None).to(DEVICE)
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_f1_macro": [],
        "val_f1_macro": [],
        "val_precision": [],
        "val_recall": [],
        "val_f1_per_class": []
    }

    scaler = GradScaler(enabled=True)

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_f1_macro.reset()

        for x_batch, y_batch in train_dl:
            x_batch = x_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.float().to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            x_batch = x_batch.unsqueeze(1).contiguous(memory_format=torch.channels_last)

            with autocast(device_type="cuda", dtype=USE_AMP_DTYPE):
                logits = model(x_batch)
                loss = criterion(logits, y_batch)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                preds = (probs > threshold).int()
                train_f1_macro.update(preds, y_batch)

        model.eval()
        val_loss = 0.0
        val_f1_macro.reset(); val_f1.reset(); val_precision.reset(); val_recall.reset()

        for x_batch, y_batch in val_dl:
            x_batch = x_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.float().to(DEVICE, non_blocking=True)
            x_batch = x_batch.unsqueeze(1).contiguous(memory_format=torch.channels_last)

            with autocast(device_type="cuda", dtype=USE_AMP_DTYPE):
                logits = model(x_batch)
                batch_loss = criterion(logits, y_batch)

            val_loss += batch_loss.item()
            preds = torch.sigmoid(logits)
            val_f1_macro.update(preds, y_batch)
            val_f1.update(preds, y_batch)
            val_precision.update(preds, y_batch)
            val_recall.update(preds, y_batch)

        scheduler.step(val_loss)

        history["train_loss"].append(train_loss / len(train_dl))
        history["val_loss"].append(val_loss / len(val_dl))

        history["train_f1_macro"].append(train_f1_macro.compute().item())
        history["val_f1_macro"].append(val_f1_macro.compute().item())

        history["val_precision"].append(val_precision.compute().cpu().numpy())
        history["val_recall"].append(val_recall.compute().cpu().numpy())
        history["val_f1_per_class"].append(val_f1.compute().cpu().numpy())

        print(f'Epoch: {epoch + 1}/{num_epochs}')

        if (epoch + 1) % save_every == 0:
            model_output_path = os.path.join(MODELS_DIRECTORY_RP, f'version_{version_num}_runpod_model_epoch{epoch + 1}.pt')
            os.makedirs(os.path.dirname(model_output_path), exist_ok=True)
            torch.save({
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scaler_state": scaler.state_dict(),
                "epoch": epoch + 1
            }, model_output_path)

    return history

In [67]:
def plot_and_save_history(history, genre_names, epochs_num, version_num, output_dir="plots"):
    os.makedirs(output_dir, exist_ok=True)
    epochs = range(epochs_num)
    # Loss
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Validation Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.title("Training vs Validation Loss")
    plt.savefig(os.path.join(output_dir, f"loss_version{version_num}.png"))
    plt.close()

    # Macro F1
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_f1_macro"], label="Train F1 Macro")
    plt.plot(epochs, history["val_f1_macro"], label="Validation F1 Macro")
    plt.xlabel("Epochs")
    plt.ylabel("F1 Score")
    plt.legend()
    plt.title("Training vs Validation Macro F1")
    plt.savefig(os.path.join(output_dir, f"macro_f1_version{version_num}.png"))
    plt.close()

    # Per-class F1
    plt.figure(figsize=(10, 6))
    for i, genre in enumerate(genre_names):
        f1_scores = [f1[i] for f1 in history["val_f1_per_class"]]
        plt.plot(epochs, f1_scores, label=genre)
    plt.xlabel("Epochs")
    plt.ylabel("F1 Score")
    plt.title("Validation F1 Scores per Class")
    plt.legend()
    plt.savefig(os.path.join(output_dir, f"f1_per_class_version{version_num}.png"))
    plt.close()

    # Per-class Recall
    plt.figure(figsize=(10, 6))
    for i, genre in enumerate(genre_names):
        recall_scores = [rec[i] for rec in history["val_recall"]]
        plt.plot(epochs, recall_scores, label=genre)
    plt.xlabel("Epochs")
    plt.ylabel("Recall")
    plt.title("Validation Recall per Class")
    plt.legend()
    plt.savefig(os.path.join(output_dir, f"recall_per_class_version{version_num}.png"))
    plt.close()

    # Per-class Precision
    plt.figure(figsize=(10, 6))
    for i, genre in enumerate(genre_names):
        precision_scores = [prec[i] for prec in history["val_precision"]]
        plt.plot(epochs, precision_scores, label=genre)
    plt.xlabel("Epochs")
    plt.ylabel("Precision")
    plt.title("Validation Precision per Class")
    plt.legend()
    plt.savefig(os.path.join(output_dir, f"precision_per_class_version{version_num}.png"))
    plt.close()

    print(f"Training history plots saved to '{output_dir}'")


In [68]:
threshold = torch.tensor([0.5, 0.3, 0.45, 0.4, 0.45, 0.4, 0.5, 0.3, 0.3, 0.5, 0.3], device=DEVICE)

In [ ]:
train_mean = torch.load(MEAN_TRAIN_RP).float()
train_std = torch.load(STD_TRAIN_RP).float()
val_mean = torch.load(MEAN_VAL_RP).float()
val_std = torch.load(STD_VAL_RP).float()

train_transform = StandardizationTransform(train_mean, train_std)
val_transform = StandardizationTransform(val_mean, val_std)

standardized_training_dataset = FeatureDataset(
    TRAIN_SPLIT_RP,
    SPECTROGRAMS_RP,
    transform=train_transform
)
standardized_val_dataset = FeatureDataset(
    VAL_SPLIT_RP,
    SPECTROGRAMS_RP,
    transform=val_transform
)

train_labels_df = load_encoded_labels_df(TRAIN_SPLIT_RP).drop('track_id', axis=1)
num_genres = get_num_genres(train_labels_df)
genre_names = train_labels_df.columns.tolist()

model = MusicGenreCNN(num_classes=num_genres)
model = model.to(DEVICE, memory_format=torch.channels_last)
try:
    model = torch.compile(model)
except Exception:
    pass

train_dl = DataLoader(
    standardized_training_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=10,
    pin_memory=True if DEVICE.type == "cuda" else False,
    persistent_workers=True,
    prefetch_factor=4,
    drop_last=True
)
val_dl = DataLoader(
    standardized_val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=10,
    pin_memory=True if DEVICE.type == "cuda" else False,
    persistent_workers=True,
    prefetch_factor=4,
    drop_last=False
)

if DEVICE.type != "cuda":
    print("CUDA not detected.")

epochs_num = 200
version_num = 7
history = train(model, epochs_num, train_dl, val_dl, num_genres, genre_names, version_num, threshold)
plot_and_save_history(history, genre_names, epochs_num, version_num, output_dir=PLOTS_RP)